# Seq2seq e atenção, na prática

Neste notebook você monta uma rede que **traduz** reclamações sobre pedidos de café para o inglês: um **encoder** lê o português e um **decoder** escreve o inglês. Depois, acrescenta **atenção** e compara.

**Não precisa anotar nada**: leia, rode e mexa no código. Rode **uma célula de cada vez**. Algumas dão erro de propósito, e isso está avisado antes delas.

## 1. Os dados

A rede traduz frases curtas do português para o inglês:

`o café quente do pedido dois cinco não estava bom` → `the hot coffee for order two five was not good`

Repare em três coisas: **café quente** vira **hot coffee** (a ordem inverte), **não estava** vira **was not** (inverte também), e **gelado**, quando aparece no fim, vira **ice cold** (uma palavra vira duas). Por isso a saída não anda junto com a entrada. O número do pedido precisa ser copiado inteiro, na ordem.

Na função `gerar`:
- `X` guarda o português, com cada palavra trocada pelo seu id.
- `Y_in` é o que o decoder **lê**: `<ini>` seguido do inglês.
- `Y_out` é o que o decoder **deve escrever**: o inglês seguido de `<fim>`.
- O que sobra no fim das linhas é `<pad>` (id 0), só para todas terem o mesmo tamanho.

In [ ]:
import numpy as np

PT = ["<pad>", "o", "café", "chá", "pão", "quente", "frio", "do", "pedido", "já", "não", "estava",
      "bom", "ruim", "gelado", "um", "dois", "três", "quatro", "cinco", "seis"]
EN = ["<pad>", "<ini>", "<fim>", "the", "coffee", "tea", "bread", "hot", "cold", "for", "order",
      "already", "not", "was", "good", "bad", "ice", "one", "two", "three", "four", "five", "six"]
ID_PT = {p: i for i, p in enumerate(PT)}
ID_EN = {p: i for i, p in enumerate(EN)}
TRADUZ = {"o": "the", "café": "coffee", "chá": "tea", "pão": "bread", "quente": "hot", "frio": "cold",
          "do": "for", "pedido": "order", "já": "already", "não": "not", "estava": "was",
          "bom": "good", "ruim": "bad", "gelado": "ice cold", "um": "one", "dois": "two", "três": "three", "quatro": "four",
          "cinco": "five", "seis": "six"}
NUMEROS = ["um", "dois", "três", "quatro", "cinco", "seis"]

def traduzir(pt):
    o, nome, adj, do, pedido, *algarismos, marca, estava, fim = pt
    return (["the", TRADUZ[adj], TRADUZ[nome], "for", "order"] + [TRADUZ[p] for p in algarismos]
            + ["was", TRADUZ[marca]] + TRADUZ[fim].split())

def frase(r, algarismos):
    return (["o", r.choice(["café", "chá", "pão"]), r.choice(["quente", "frio"]), "do", "pedido"]
            + list(r.choice(NUMEROS, size=algarismos))
            + [r.choice(["já", "não"]), "estava", r.choice(["bom", "ruim", "gelado"])])

def gerar(n, algarismos, seed):
    r = np.random.default_rng(seed)
    T = algarismos + 10                     # a frase em inglês, mais <ini> ou <fim>
    X = np.zeros((n, algarismos + 8), int)   # português, em ids
    Y_in = np.zeros((n, T), int)           # o que o decoder lê: <ini> + inglês
    Y_out = np.zeros((n, T), int)          # o que o decoder deve escrever: inglês + <fim>
    for i in range(n):
        pt = frase(r, algarismos)
        en = traduzir(pt)
        X[i] = [ID_PT[p] for p in pt]
        Y_in[i, :len(en) + 1] = [ID_EN[p] for p in ["<ini>"] + en]
        Y_out[i, :len(en) + 1] = [ID_EN[p] for p in en + ["<fim>"]]
    return X, Y_in, Y_out

ALGARISMOS = 6   # quantos algarismos tem o número do pedido
X_tr, Y_in_tr, Y_out_tr = gerar(3000, ALGARISMOS, 1)
X_val, Y_in_val, Y_out_val = gerar(500, ALGARISMOS, 2)
X_teste, Y_in_teste, Y_out_teste = gerar(500, ALGARISMOS, 3)

print(X_tr.shape, Y_in_tr.shape, Y_out_tr.shape)
print(" ".join(PT[i] for i in X_tr[0]))
print(" ".join(EN[i] for i in Y_out_tr[0] if i != 0))

`Y_in` e `Y_out` são a mesma frase, deslocada de uma posição. Em cada passo, o decoder lê uma palavra e deve escrever a seguinte:

In [ ]:
for lida, escrita in zip(Y_in_tr[0], Y_out_tr[0]):
    if escrita == 0:
        break
    print(f"lê {EN[lida]:>9}  ->  escreve {EN[escrita]}")

## 2. Construir, uma camada por vez

Cada camada recebe a saída da anterior. Olhe o formato que sai de cada uma. O lote são 4 frases da validação.

In [ ]:
import keras
from keras import layers, Input, Model
import numpy as np

pt4, in4, out4 = X_val[:4], Y_in_val[:4], Y_out_val[:4]

# Encoder: Embedding troca cada id por um vetor de 16 números; a LSTM lê as 14 palavras em português.
v_pt = layers.Embedding(len(PT), 16)(pt4)
ultimo, h, c = layers.LSTM(32, return_state=True)(v_pt)
print("português em vetores", v_pt.shape)
print("estado do encoder   ", h.shape, c.shape)

Com `return_state=True`, a LSTM devolve também o estado final: `h` e `c`, os dois vetores da LSTM. **É tudo o que o encoder entrega ao decoder**: 32 + 32 números para a frase inteira.

In [ ]:
# Decoder: outra Embedding (o vocabulário é o inglês) e outra LSTM, que começa do estado do encoder.
v_en = layers.Embedding(len(EN), 16)(in4)
estados_en = layers.LSTM(32, return_sequences=True)(v_en, initial_state=[h, c])
print("inglês em vetores ", v_en.shape)
print("estados do decoder", estados_en.shape)

# Dense: em cada posição, uma probabilidade para cada uma das 23 palavras do inglês.
prob = layers.Dense(len(EN), activation="softmax")(estados_en)
print("probabilidades    ", prob.shape)
print("soma da primeira posição", float(prob[0, 0].numpy().sum()))

## 3. Encoder e decoder de tamanhos diferentes

A célula abaixo **dá erro de propósito**. Leia a mensagem: ela mostra os tamanhos das duas matrizes que não se encaixam.

In [ ]:
estados_en = layers.LSTM(64, return_sequences=True)(v_en, initial_state=[h, c])

O decoder começa do estado do encoder, então os dois precisam ter o mesmo número de unidades. Corrija a célula acima.

## 4. O modelo inteiro

O mesmo encoder e decoder, agora como um modelo com duas entradas: o português e o inglês deslocado.

A perda usa `ignore_class=0`: as posições com `<pad>` não contam. O otimizador é o Adam com taxa de aprendizado 0.003 (o padrão é 0.001; nesta tarefa, com o padrão, o modelo com atenção às vezes demora muito a aprender).

In [ ]:
def montar_sem_atencao(unidades=32, dim=16):
    ent_pt = Input(shape=(None,), name="portugues")
    ent_en = Input(shape=(None,), name="ingles_deslocado")
    v_pt = layers.Embedding(len(PT), dim)(ent_pt)
    v_en = layers.Embedding(len(EN), dim)(ent_en)
    _, h, c = layers.LSTM(unidades, return_state=True, name="encoder")(v_pt)
    estados_en = layers.LSTM(unidades, return_sequences=True, name="decoder")(v_en, initial_state=[h, c])
    saida = layers.Dense(len(EN), activation="softmax")(estados_en)
    modelo = Model([ent_pt, ent_en], saida)
    modelo.compile(keras.optimizers.Adam(0.003), keras.losses.SparseCategoricalCrossentropy(ignore_class=0))
    return modelo

keras.utils.set_random_seed(0)
sem = montar_sem_atencao()
sem.summary()

## 5. Treinar

No treino, o decoder lê a frase certa em inglês (`Y_in`), mesmo quando ele próprio erraria a palavra anterior. Leva cerca de um minuto e meio.

In [ ]:
hist_sem = sem.fit([X_tr, Y_in_tr], Y_out_tr, epochs=20, batch_size=32,
                   validation_data=([X_val, Y_in_val], Y_out_val), verbose=2)

## 6. Gerar não é treinar

Para traduzir uma frase nova, não existe `Y_in`: ninguém dá a resposta. O decoder começa com `<ini>`, escolhe a palavra mais provável, e essa palavra **volta como entrada** do passo seguinte. Para em `<fim>`.

In [ ]:
INI, FIM = ID_EN["<ini>"], ID_EN["<fim>"]

def gerar_traducao(modelo, X, passos=Y_out_tr.shape[1]):
    Y = np.full((len(X), 1), INI)
    for _ in range(passos):
        p = modelo.predict([X, Y], verbose=0, batch_size=512)[:, -1]   # a próxima palavra
        Y = np.concatenate([Y, p.argmax(-1)[:, None]], axis=1)          # volta como entrada
    return Y[:, 1:]

def texto(ids):
    palavras = []
    for i in ids:
        if i == FIM:
            break
        palavras.append(EN[i])
    return " ".join(palavras)

def mostrar(modelo, X, Y_out, n=4):
    G = gerar_traducao(modelo, X[:n])
    for x, g, y in zip(X[:n], G, Y_out[:n]):
        print("português:", " ".join(PT[i] for i in x))
        print("esperado: ", texto(y))
        print("gerado:   ", texto(g), "\n")

mostrar(sem, X_val, Y_out_val)

Duas medidas diferentes, na validação:
- **palavras, com a resposta dada**: o decoder lê a frase certa e acerta a próxima palavra? É o que o treino otimiza.
- **frases geradas inteiras**: o decoder gera sozinho, e a frase só conta se sair igual, palavra por palavra, até o `<fim>`.

In [ ]:
def acerto_palavras(modelo, X, Y_in, Y_out):
    p = modelo.predict([X, Y_in], verbose=0, batch_size=512).argmax(-1)
    conta = Y_out != 0
    return (p[conta] == Y_out[conta]).mean()

def acerto_frases(modelo, X, Y_out):
    G = gerar_traducao(modelo, X)
    certas = [texto(g) == texto(y) for g, y in zip(G, Y_out)]
    return np.mean(certas)

print("palavras, com a resposta dada:", round(acerto_palavras(sem, X_val, Y_in_val, Y_out_val), 3))
print("frases geradas inteiras:      ", round(acerto_frases(sem, X_val, Y_out_val), 3))

## 7. Uma linha a mais: atenção

Sem atenção, o decoder só conhece o português pelo estado final `h, c`. Com atenção, **em cada passo** o decoder compara o seu estado com o estado do encoder **em cada palavra** do português, transforma as notas em pesos que somam 1 e mistura esses estados com os pesos.

Em Keras: `layers.Attention()([consulta, valor])`. A **consulta** é o estado do decoder; o **valor** são os estados do encoder.

A célula abaixo **dá erro de propósito**: ela passa só o estado final do encoder.

In [ ]:
contexto = layers.Attention()([estados_en, h])

A atenção precisa de um vetor **por palavra** do português, e o encoder entregou um só. Na célula abaixo, o encoder devolve os estados de todas as palavras (`return_sequences=True`). Rode e olhe os formatos.

In [ ]:
estados_pt, h, c = layers.LSTM(32, return_sequences=True, return_state=True)(v_pt)
estados_en = layers.LSTM(32, return_sequences=True)(v_en, initial_state=[h, c])
contexto, pesos = layers.Attention()([estados_en, estados_pt], return_attention_scores=True)

print("estados do encoder", estados_pt.shape)   # um vetor por palavra do português
print("estados do decoder", estados_en.shape)   # um vetor por posição do inglês
print("pesos             ", pesos.shape)        # para cada posição do inglês, um peso por palavra do português
print("contexto          ", contexto.shape)     # para cada posição do inglês, a mistura dos estados do encoder
print("soma de uma linha de pesos:", float(pesos[0, 0].numpy().sum()))

Agora o modelo inteiro. Compare com `montar_sem_atencao`: muda o encoder (`return_sequences=True`) e entram duas linhas, a atenção e a junção do contexto com o estado do decoder.

In [ ]:
def montar_com_atencao(unidades=32, dim=16):
    ent_pt = Input(shape=(None,), name="portugues")
    ent_en = Input(shape=(None,), name="ingles_deslocado")
    v_pt = layers.Embedding(len(PT), dim)(ent_pt)
    v_en = layers.Embedding(len(EN), dim)(ent_en)
    estados_pt, h, c = layers.LSTM(unidades, return_sequences=True, return_state=True, name="encoder")(v_pt)
    estados_en = layers.LSTM(unidades, return_sequences=True, name="decoder")(v_en, initial_state=[h, c])
    contexto, pesos = layers.Attention(name="atencao")([estados_en, estados_pt], return_attention_scores=True)
    juntos = layers.Concatenate()([estados_en, contexto])
    saida = layers.Dense(len(EN), activation="softmax")(juntos)
    modelo = Model([ent_pt, ent_en], saida)
    modelo.compile(keras.optimizers.Adam(0.003), keras.losses.SparseCategoricalCrossentropy(ignore_class=0))
    return modelo, Model([ent_pt, ent_en], pesos)

keras.utils.set_random_seed(0)
com, mapa = montar_com_atencao()
com.summary()
print("sem atenção:", sem.count_params(), " com atenção:", com.count_params())

Olhe a linha da camada `atencao` no resumo: quantos parâmetros ela tem? De onde vem a diferença no total?

## 8. Treinar com atenção e comparar

Mesmos dados, mesma seed, mesmas 20 épocas.

In [ ]:
hist_com = com.fit([X_tr, Y_in_tr], Y_out_tr, epochs=20, batch_size=32,
                   validation_data=([X_val, Y_in_val], Y_out_val), verbose=2)
mostrar(com, X_val, Y_out_val)
for nome, modelo in [("sem atenção", sem), ("com atenção", com)]:
    print(nome, " palavras:", round(acerto_palavras(modelo, X_val, Y_in_val, Y_out_val), 3),
          " frases:", round(acerto_frases(modelo, X_val, Y_out_val), 3))

## 9. Para onde o decoder olha

O modelo `mapa` devolve os pesos da atenção. Cada linha do desenho é uma palavra que o decoder escreveu; cada coluna, uma palavra do português. Quanto mais escuro, maior o peso.

In [ ]:
import matplotlib.pyplot as plt

def desenhar_mapa(i):
    x = X_val[i:i + 1]
    g = gerar_traducao(com, x)[0]
    escrito = texto(g).split() + ["<fim>"]
    lido = np.array([[INI] + [ID_EN[p] for p in escrito[:-1]]])
    p = mapa.predict([x, lido], verbose=0)[0]
    plt.figure(figsize=(7, 5))
    plt.imshow(p, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(x.shape[1]), [PT[j] for j in x[0]], rotation=60)
    plt.yticks(range(len(escrito)), escrito)
    plt.xlabel("português (o que o encoder leu)")
    plt.ylabel("inglês (o que o decoder escreveu)")
    plt.colorbar(label="peso")
    plt.show()

desenhar_mapa(0)

Procure as linhas dos algarismos do pedido, e as de **hot** e **coffee** (ou cold, tea, bread). Troque o número em `desenhar_mapa(0)` para ver outras frases.

## 10. Um bug que não dá erro

Na célula abaixo, alguém treinou passando `Y_out` no lugar de `Y_in`. Não há erro. Rode, olhe a perda, e depois gere traduções com esse modelo.

In [ ]:
keras.utils.set_random_seed(0)
bug, _ = montar_com_atencao()
bug.fit([X_tr, Y_out_tr], Y_out_tr, epochs=20, batch_size=32,
        validation_data=([X_val, Y_out_val], Y_out_val), verbose=2)

In [ ]:
mostrar(bug, X_val, Y_out_val)

Compare com a célula da seção 1 que mostra `lê -> escreve`. O que o decoder lia em cada passo, nesse treino com bug?

## 11. Para mexer

- Mude `ALGARISMOS` na célula 1 para 2 e rode de novo os dois modelos. E para 10.
- Mude o número de unidades dos dois modelos.
- No modelo com atenção, tire o `initial_state=[h, c]` do decoder. O decoder ainda consegue saber qual é a frase?